# 2048 AI - Treino DQN no Google Colab

Este notebook treina o primeiro algoritmo de aprendizagem do projeto: um Double DQN basico. O treino padrao e moderado para ja permitir avaliacao, sem exigir um treino longo logo de inicio.

## 1. Preparar ambiente

Use CPU para validacao curta. GPU pode ser ativada depois em `Ambiente de execucao > Alterar tipo de ambiente de execucao`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Sankofa-JBC/2048-ai.git"
PROJECT_DIR = Path("/content/2048-ai")
SRC_DIR = PROJECT_DIR / "src"

if PROJECT_DIR.exists():
    print("Repositorio ja existe. Atualizando...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)
else:
    print("Clonando repositorio...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

if not (PROJECT_DIR / "pyproject.toml").exists():
    raise FileNotFoundError("pyproject.toml nao encontrado na raiz do projeto clonado.")

print("Instalando projeto e dependencias de aprendizagem...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[learning]"],
    cwd=PROJECT_DIR,
    check=True,
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Raiz do projeto:", PROJECT_DIR)

## 2. Conferir PyTorch e imports

Se esta celula falhar, o ambiente ainda nao esta pronto para treino.

In [ ]:
import torch
from game2048.learning.training import DQNTrainingConfig, train_dqn

print("Torch:", torch.__version__)
print("CUDA disponivel:", torch.cuda.is_available())

## 3. Rodar testes

Antes do treino, confirmamos que o jogo, agentes e avaliador continuam funcionando no Colab.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-t", "."],
    cwd=PROJECT_DIR,
    check=True,
)

## 4. Treino inicial

Este treino ja salva checkpoint final, melhor checkpoint por media movel e historico de metricas em JSON.

In [ ]:
subprocess.run([
    sys.executable,
    "train_dqn.py",
    "--episodes",
    "300",
    "--seed",
    "42",
    "--batch-size",
    "64",
    "--min-replay-size",
    "500",
    "--memory-capacity",
    "20000",
    "--checkpoint-interval",
    "100",
    "--rolling-window",
    "50",
    "--metrics-output",
    "results/dqn_training_metrics.json",
    "--best-output",
    "models/dqn_2048_best.pt",
    "--output",
    "models/dqn_2048.pt",
], cwd=PROJECT_DIR, check=True)

## 5. Avaliar o melhor checkpoint DQN

Agora avaliamos o melhor modelo salvo durante o treino. Em treinos curtos, ele ainda pode perder para o heuristico.

In [ ]:
subprocess.run([
    sys.executable,
    "evaluate_dqn.py",
    "models/dqn_2048_best.pt",
    "--games",
    "100",
    "--seed",
    "42",
    "--output",
    "results/dqn_best_eval.json",
], cwd=PROJECT_DIR, check=True)

## 6. Comparar com baselines

Comparamos rapidamente contra random e heuristic para ter contexto.

In [ ]:
subprocess.run([sys.executable, "evaluate_agents.py", "--agent", "random", "--games", "100", "--seed", "42"], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, "evaluate_agents.py", "--agent", "heuristic", "--games", "100", "--seed", "42"], cwd=PROJECT_DIR, check=True)

## 7. Ver metricas de treino

Esta celula mostra os ultimos episodios do historico salvo em JSON.

In [ ]:
import json

with open(PROJECT_DIR / "results/dqn_training_metrics.json", "r", encoding="utf-8") as file:
    metrics = json.load(file)

print("Melhor media movel:", round(metrics["best_rolling_average_score"], 2))
print("Melhor checkpoint:", metrics["best_save_path"])
print("Ultimos episodios:")
for item in metrics["episodes"][-10:]:
    print(
        item["episode"],
        "score=", item["score"],
        "media=", round(item["rolling_average_score"], 2),
        "tile=", item["max_tile"],
        "epsilon=", round(item["epsilon"], 3),
    )

## 8. Baixar checkpoint opcionalmente

Use esta celula quando quiser baixar o melhor modelo treinado para o PC.

In [ ]:
# Descomente as duas linhas abaixo no Colab para baixar o melhor checkpoint.
# from google.colab import files
# files.download(str(PROJECT_DIR / "models/dqn_2048_best.pt"))

## Proximo passo

Depois de validar esse treino, podemos aumentar episodios para 1000, 3000 ou mais e comparar a evolucao contra random e heuristic.